# Multimodal Visual Question Answering Transformer

This notebook demonstrates **pretrained ViLT inference** and VQA-style evaluation.
It does not claim model fine-tuning. The public deployment is a separate browser-only
Static Space using Transformers.js.

> Use only safe, non-sensitive images. Model answers and confidence proxies can be wrong.


In [ ]:
from pathlib import Path
import sys, json, pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vqa.dataset_loader import load_vqa_csv
from vqa.question_preprocessing import preprocess_question, classify_question_type
from vqa.image_preprocessing import load_and_validate_image
from vqa.evaluation import evaluate_records


## 1. Inspect the safe sample dataset

In [ ]:
records = load_vqa_csv(PROJECT_ROOT / "data/sample_vqa_pairs.csv")
sample_df = pd.DataFrame(records)
sample_df

## 2. Validate preprocessing

In [ ]:
question = preprocess_question(records[0]["question"])
image = load_and_validate_image(PROJECT_ROOT / records[0]["image_path"])
print(question, classify_question_type(question), image.size, image.mode)

## 3. Load ViLT and run inference

This cell downloads `dandelin/vilt-b32-finetuned-vqa` from the Hugging Face Hub.
Run it only after installing `requirements.txt`.


In [ ]:
from vqa.inference_pipeline import VQAInferencePipeline

pipeline = VQAInferencePipeline()
result = pipeline.predict(PROJECT_ROOT / records[0]["image_path"], records[0]["question"])
result.to_dict()

## 4. Evaluate the documented sample

In [ ]:
predictions = []
for row in records:
    prediction = pipeline.predict(PROJECT_ROOT / row["image_path"], row["question"])
    predictions.append({**row, "prediction": prediction.answer})
metrics = evaluate_records(predictions)
metrics

## 5. Interpretation

The three synthetic records are smoke tests, not a benchmark. Run the evaluation
script against a documented VQA v2 subset before publishing accuracy. Review failures
manually and report hardware-specific latency separately.
